<a href="https://colab.research.google.com/github/HimanshuPathak2725/googleColab/blob/main/sac_numpy_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Constructing Soft Actor-Critic from First Principles: An Inquiry in NumPy

Can we build a fully operational, state-of-the-art continuous reinforcement learning agent without relying on modern deep learning frameworks? This notebook is a rigorous exercise in scientific reconstruction.

We will implement the entire mathematical machinery of **Soft Actor-Critic (SAC)**—including reverse-mode automatic differentiation (backpropagation), the Adam optimizer, replay memories, twin value networks, target networks, a squashed-Gaussian stochastic policy, and dynamic entropy-temperature tuning—using nothing but **NumPy and the Python standard library**.

### Why This Inquiry?
Modern libraries like PyTorch and JAX hide the physical reality of gradients, linear algebra, and tracking variables over time. By stripping away these abstractions, we force ourselves to look directly at the underlying mathematics: how errors propagate backwards, how stochastic policies navigate high-dimensional spaces, and how value landscapes stabilize. This is not just an engineering task; it is an investigation into how artificial systems learn to control their environments.

## 1. The Mathematical Foundations: Why Entropy Stabilizes Learning

To understand why we are writing this specific code, we must first unpack the mathematical engine of Soft Actor-Critic.

### The Maximum-Entropy Objective
In standard Reinforcement Learning, our goal is simply to maximize the expected sum of discounted future rewards:

$$
J(\pi)=\mathbb E_{\tau\sim\pi}
\left[\sum_{t=0}^{\infty}\gamma^t r_t\right].
$$

However, this pure exploitation objective often leads to brittle, deterministic policies that collapse into sub-optimal local minima. SAC solves this by redefining what it means to be successful. We augment the reward function with a term that actively encourages exploration: the entropy of the policy itself. The agent is now rewarded for both achieving goals and maintaining choices:

$$
J(\pi)=\mathbb E\left[\sum_{t=0}^\infty \gamma^t\left(r_t+\alpha\mathcal H(\pi(\cdot\mid s_t))\right)\right],
\qquad \mathcal H(\pi(\cdot\mid s))=-\mathbb E_{a\sim\pi}[\log\pi(a\mid s)].
$$

Here, the temperature parameter $\alpha > 0$ dictates the trade-off. It governs how much the agent values immediate reward versus keeping its future options open. Every single step we sample will contribute `reward - alpha * log_prob` to our learning signal.

### The Soft Bellman Equations
To evaluate policies under this maximum-entropy objective, we modify our value functions. The soft state-value function $V^\pi(s)$ must capture both future rewards and future entropy:

$$
V^\pi(s)=\mathbb E_{a\sim\pi}\left[Q^\pi(s,a)-\alpha\log\pi(a\mid s)\right],
$$

and the soft action-value function $Q^\pi(s,a)$ is computed as standard temporal-difference expectations over the next soft state-value:

$$
Q^\pi(s,a)=\mathbb E_{s'\sim p}\left[r(s,a)+\gamma V^\pi(s')\right].
$$

Substituting $V$ into $Q$ gives us the core recursive equation we will implement in our training loop:

$$
Q^\pi(s,a)=\mathbb E\left[r+\gamma\left(Q^\pi(s',a')-\alpha\log\pi(a'\mid s')\right)\right].
$$

### Overcoming Bias: Twin Critics and Target Networks
In value-based RL, the max operator in Bellman updates systematically overestimates action values. To combat this mathematically, we deploy two independent critics ($Q_1$ and $Q_2$). When calculating targets, we take the minimum of their predictions:

$$
y=r+\gamma(1-d)\left[\min_{i\in\{1,2\}}\bar Q_{\bar\phi_i}(s',a')-\alpha\log\pi_\theta(a'\mid s')\right],
$$

where $a'$ is sampled from our current policy $\pi_\theta$, and $d$ represents terminal transitions. We will train each critic by minimizing the squared Bellman error against this shared target:

$$
\mathcal L_{Q_i}=\frac12\mathbb E_{\mathcal D}\left[(Q_{\phi_i}(s,a)-\text{stopgrad}(y))^2\right].
$$

To keep our training targets stable, we slowly track the main critic parameters using target networks ($\bar{Q}$), updated via Polyak averaging:

$$
\bar\phi_i\leftarrow(1-\tau)\bar\phi_i+\tau\phi_i.
$$

### The Squashed-Gaussian Policy and the Change-of-Variables Theorem
Our actor must output actions within bounded physical limits (e.g., $[-1, 1]$). Simply clipping a normal distribution breaks gradient propagation. Instead, our neural network will output the mean $\mu_\theta(s)$ and log standard deviation $\ell_\theta(s)$ of an unbounded Gaussian, which we then squash through a $\tanh$ function:

$$
\epsilon\sim\mathcal N(0,I),\quad u=\mu_\theta(s)+e^{\ell_\theta(s)}\odot\epsilon,\quad h=\tanh(u),\quad a=b+c\odot h,
$$

where $b$ is the action offset and $c$ is the scale. Because we are transforming a probability density function through a non-linear mapping, we must apply the Change-of-Variables theorem. The squashing operation compresses probability mass near the boundaries, so we mathematically correct the log probability density:

$$
\log\pi_\theta(a\mid s)=\sum_j[-\tfrac12\epsilon_j^2-\ell_j-\tfrac12\log(2\pi)-\log(1-h_j^2+\delta)-\log c_j].
$$

This correction is vital; without it, our actor's calculations of its own entropy would be fundamentally distorted, breaking the policy updates.

### Dynamically Tuning Exploration: Automatic Entropy Tuning
Manually tuning the temperature $\alpha$ is incredibly difficult because the scale of rewards changes as the agent learns. Instead, we treat the optimization of $\alpha$ as a dual problem with a target constraint $\mathcal{H}_*$. By adjusting $\beta = \log \alpha$, we can automatically adjust our exploration drive:

$$
\mathcal L_\alpha=-\mathbb E\left[\beta\,\text{stopgrad}(\log\pi_\theta(a\mid s)+\mathcal H_*)\right],
$$

which yields the gradient:

$$
\nabla_\beta\mathcal L_\alpha=-\mathbb E[\log\pi_\theta+\mathcal H_*]=\mathcal H_\pi-\mathcal H_*.
$$

If the policy's real entropy $\mathcal H_\pi$ drops below the threshold $\mathcal H_*$, the gradient becomes negative, driving $\alpha$ up to force more exploration. If the agent is exploring too wildly, $\alpha$ is driven down to favor exploitation.

## 2. Bootstrapping Our Workspace

Before writing our neural layers, we establish our scientific instruments. We import only standard Python libraries and standard NumPy. By refusing external deep learning frameworks, we ensure that every matrix multiplication and derivative tracking is fully visible.

In [1]:
from __future__ import annotations
import copy
import csv
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Tuple

import numpy as np

LOG_2PI = math.log(2.0 * math.pi)


## 3. Implementing the Engine: Deep Learning from Scratch

To train our actor and critics, we need a mechanism to compute gradients. We will implement our own fully-connected Multi-Layer Perceptron (MLP) complete with manual reverse-mode automatic differentiation and a clean Adam optimizer.

Our MLP must keep track of intermediate values (such as pre-activations and hidden layers) in a `cache` during the forward pass. This is because backpropagation requires these stored activations to calculate exact local derivatives. During actor updates, we also need to pass gradients backward through our critics to adjust our actions—all while keeping the critic parameters frozen. This requires a carefully designed `backward` pass.

In [2]:
class Adam:
    """Minimal Adam optimizer for a dictionary of NumPy parameter arrays."""
    def __init__(self, params: Dict[str, np.ndarray], lr: float = 3e-4,
                 beta1: float = 0.9, beta2: float = 0.999, eps: float = 1e-8):
        self.lr, self.beta1, self.beta2, self.eps = lr, beta1, beta2, eps
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}
        self.t = 0

    def step(self, params: Dict[str, np.ndarray], grads: Dict[str, np.ndarray]) -> None:
        self.t += 1
        b1_correction = 1.0 - self.beta1 ** self.t
        b2_correction = 1.0 - self.beta2 ** self.t
        for key, param in params.items():
            grad = grads[key]
            self.m[key] = self.beta1 * self.m[key] + (1.0 - self.beta1) * grad
            self.v[key] = self.beta2 * self.v[key] + (1.0 - self.beta2) * (grad * grad)
            m_hat = self.m[key] / b1_correction
            v_hat = self.v[key] / b2_correction
            param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


class MLP:
    """Fully-connected MLP with ReLU hidden layers and a linear output."""
    def __init__(self, sizes, rng: np.random.Generator, lr: float = 3e-4,
                 output_bias: np.ndarray | None = None, small_output: bool = False):
        self.params: Dict[str, np.ndarray] = {}
        self.num_layers = len(sizes) - 1
        for i, (din, dout) in enumerate(zip(sizes[:-1], sizes[1:])):
            if i == self.num_layers - 1 and small_output:
                scale = 0.01
            else:
                scale = math.sqrt(2.0 / din) if i < self.num_layers - 1 else math.sqrt(1.0 / din)
            self.params[f"W{i}"] = rng.standard_normal((din, dout)) * scale
            self.params[f"b{i}"] = np.zeros(dout)
        if output_bias is not None:
            self.params[f"b{self.num_layers - 1}"][:] = output_bias
        self.optimizer = Adam(self.params, lr=lr)

    def forward(self, x: np.ndarray) -> Tuple[np.ndarray, tuple]:
        activations = [x]
        preactivations = []
        h = x
        for i in range(self.num_layers):
            z = h @ self.params[f"W{i}"] + self.params[f"b{i}"]
            preactivations.append(z)
            if i < self.num_layers - 1:
                h = np.maximum(z, 0.0)
            else:
                h = z
            activations.append(h)
        return h, (activations, preactivations)

    def backward(self, cache: tuple, grad_out: np.ndarray, update: bool = True) -> np.ndarray:
        activations, preactivations = cache
        grad = grad_out
        grads: Dict[str, np.ndarray] = {}
        for i in reversed(range(self.num_layers)):
            grads[f"W{i}"] = activations[i].T @ grad
            grads[f"b{i}"] = np.sum(grad, axis=0)
            grad = grad @ self.params[f"W{i}"].T
            if i > 0:
                grad = grad * (preactivations[i - 1] > 0.0)
        grad_input = grad
        if update:
            self.optimizer.step(self.params, grads)
        return grad_input


## 4. Designing the Physical Arena and Replay Memory

To test our mathematical derivation, we need a physical sandbox. We will construct a continuous-control environment from first principles: a damped one-dimensional point mass.

Our agent controls the acceleration $a \in [-1, 1]$. Its goal is to navigate to the origin ($x=0$) and stabilize there with minimum velocity ($v=0$) using minimal physical effort. The instantaneous reward at any moment is defined as:

$$
r_t=-(x_t^2+0.08v_t^2+0.01a_t^2).
$$

### The Nature of Time Limits and Replay Memories
We must also construct a `ReplayBuffer` to store and shuffle our experiences. When storing transitions, we make an important distinction: if the episode ends due to a physical failure (falling off the track, `terminated`), the true state-value is zero, and we stop bootstrapping. If it ends simply because the session timed out (`truncated`), the agent could have kept going. We must continue bootstrapping from the final state, preserving the mathematical integrity of our infinite-horizon objective.

In [3]:
class PointMassEnv:
    """Custom 1-D damped point mass; no Gym or other environment library.

    State: normalized position and velocity. Action: acceleration in [-1, 1].
    Objective: keep the point at x=0 with low velocity and modest effort.
    """
    def __init__(self, seed: int = 0, horizon: int = 100):
        self.rng = np.random.default_rng(seed)
        self.horizon = horizon
        self.action_low = np.array([-1.0])
        self.action_high = np.array([1.0])
        self.t = 0
        self.x = 0.0
        self.v = 0.0

    def _obs(self) -> np.ndarray:
        return np.array([self.x / 2.5, self.v / 2.0], dtype=np.float64)

    def reset(self) -> np.ndarray:
        self.x = float(self.rng.uniform(-1.5, 1.5))
        self.v = float(self.rng.uniform(-0.5, 0.5))
        self.t = 0
        return self._obs()

    def step(self, action: np.ndarray):
        a = float(np.clip(np.asarray(action).reshape(-1)[0], -1.0, 1.0))
        self.v = 0.90 * self.v + 0.20 * a
        self.x = self.x + 0.10 * self.v
        self.t += 1
        reward = -(self.x * self.x + 0.08 * self.v * self.v + 0.01 * a * a)
        terminated = abs(self.x) > 2.5
        truncated = self.t >= self.horizon
        return self._obs(), float(reward), bool(terminated), bool(truncated)


class ReplayBuffer:
    def __init__(self, capacity: int, state_dim: int, action_dim: int, rng: np.random.Generator):
        self.capacity = capacity
        self.rng = rng
        self.states = np.zeros((capacity, state_dim))
        self.actions = np.zeros((capacity, action_dim))
        self.rewards = np.zeros((capacity, 1))
        self.next_states = np.zeros((capacity, state_dim))
        self.dones = np.zeros((capacity, 1))
        self.pos = 0
        self.size = 0

    def add(self, s, a, r, s2, terminated):
        i = self.pos
        self.states[i] = s
        self.actions[i] = a
        self.rewards[i, 0] = r
        self.next_states[i] = s2
        # True MDP termination stops bootstrapping. Time-limit truncation does not.
        self.dones[i, 0] = float(terminated)
        self.pos = (self.pos + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size: int):
        idx = self.rng.integers(0, self.size, size=batch_size)
        return (self.states[idx], self.actions[idx], self.rewards[idx],
                self.next_states[idx], self.dones[idx])


@dataclass
class SACConfig:
    gamma: float = 0.99
    tau: float = 0.005
    lr: float = 3e-4
    batch_size: int = 64
    hidden: int = 32
    replay_size: int = 100_000
    learning_starts: int = 500
    target_entropy_per_dim: float = 0.5
    log_std_min: float = -5.0
    log_std_max: float = 2.0


## 5. Assembling the SAC Agent

Here, we synthesize our mathematical components into a single agent. The `SAC` class integrates our MLP networks, manages the squashed-Gaussian distribution, computes the pathwise policy gradients, calculates soft temporal-difference targets, and dynamically adjusts the temperature parameter $\alpha$. Every line in this class maps directly to the mathematical equations we derived in Section 1.

In [4]:
class SAC:
    def __init__(self, state_dim: int, action_low: np.ndarray, action_high: np.ndarray,
                 seed: int = 0, config: SACConfig | None = None):
        self.cfg = config or SACConfig()
        self.rng = np.random.default_rng(seed)
        self.state_dim = state_dim
        self.action_low = np.asarray(action_low, dtype=np.float64)
        self.action_high = np.asarray(action_high, dtype=np.float64)
        self.action_dim = int(self.action_low.size)
        self.action_scale = (self.action_high - self.action_low) / 2.0
        self.action_bias = (self.action_high + self.action_low) / 2.0
        self.target_entropy = self.cfg.target_entropy_per_dim * self.action_dim

        actor_bias = np.concatenate([np.zeros(self.action_dim), np.full(self.action_dim, -0.5)])
        self.actor = MLP([state_dim, self.cfg.hidden, self.cfg.hidden, 2 * self.action_dim],
                         self.rng, lr=self.cfg.lr, output_bias=actor_bias, small_output=True)
        critic_sizes = [state_dim + self.action_dim, self.cfg.hidden, self.cfg.hidden, 1]
        self.q1 = MLP(critic_sizes, self.rng, lr=self.cfg.lr)
        self.q2 = MLP(critic_sizes, self.rng, lr=self.cfg.lr)
        self.q1_target = copy.deepcopy(self.q1)
        self.q2_target = copy.deepcopy(self.q2)
        # Target networks are inference-only; their copied optimizers are unused.
        self.log_alpha = 0.0  # alpha starts at 1
        self.alpha_m = 0.0
        self.alpha_v = 0.0
        self.alpha_t = 0
        self.replay = ReplayBuffer(self.cfg.replay_size, state_dim, self.action_dim, self.rng)

    @property
    def alpha(self) -> float:
        return float(np.exp(self.log_alpha))

    def load_checkpoint(self, path: str) -> None:
        """Load exported parameters for inference (Adam state is not checkpointed)."""
        networks = (("actor", self.actor), ("q1", self.q1), ("q2", self.q2),
                    ("q1_target", self.q1_target), ("q2_target", self.q2_target))
        with np.load(path) as data:
            self.log_alpha = float(data["log_alpha"])
            for prefix, network in networks:
                for key, value in network.params.items():
                    name = f"{prefix}_{key}"
                    if name not in data or data[name].shape != value.shape:
                        raise ValueError(f"Checkpoint parameter missing or shape mismatch: {name}")
                    value[...] = data[name]

    def _actor_sample(self, states: np.ndarray):
        out, cache = self.actor.forward(states)
        mu = out[:, :self.action_dim]
        raw_log_std = out[:, self.action_dim:]
        log_std = np.clip(raw_log_std, self.cfg.log_std_min, self.cfg.log_std_max)
        clip_mask = ((raw_log_std >= self.cfg.log_std_min) &
                     (raw_log_std <= self.cfg.log_std_max)).astype(np.float64)
        std = np.exp(log_std)
        eps = self.rng.standard_normal(mu.shape)
        z = mu + std * eps
        squashed = np.tanh(z)
        actions = self.action_bias + self.action_scale * squashed
        # Change-of-variables correction for a tanh squashing and action scaling.
        normal_logp = -0.5 * eps * eps - log_std - 0.5 * LOG_2PI
        tanh_correction = np.log(1.0 - squashed * squashed + 1e-6)
        scale_correction = np.log(self.action_scale + 1e-12)
        logp = np.sum(normal_logp - tanh_correction - scale_correction, axis=1, keepdims=True)
        # d[-log(1-tanh(z)^2 + eps)] / dz
        d_logp_dz = 2.0 * squashed * (1.0 - squashed * squashed) / (1.0 - squashed * squashed + 1e-6)
        dlogp_dmu = d_logp_dz
        dlogp_dlogstd = -np.ones_like(log_std) + d_logp_dz * std * eps
        return {
            "actions": actions, "logp": logp, "cache": cache, "mu": mu,
            "raw_log_std": raw_log_std, "log_std": log_std, "clip_mask": clip_mask,
            "std": std, "eps": eps, "z": z, "squashed": squashed,
            "dlogp_dmu": dlogp_dmu, "dlogp_dlogstd": dlogp_dlogstd,
        }

    def act(self, state: np.ndarray, deterministic: bool = False) -> np.ndarray:
        out, _ = self.actor.forward(np.asarray(state, dtype=np.float64).reshape(1, -1))
        mu = out[:, :self.action_dim]
        if deterministic:
            unit_action = np.tanh(mu)
        else:
            log_std = np.clip(out[:, self.action_dim:], self.cfg.log_std_min, self.cfg.log_std_max)
            unit_action = np.tanh(mu + np.exp(log_std) * self.rng.standard_normal(mu.shape))
        action = self.action_bias + self.action_scale * unit_action
        return np.clip(action[0], self.action_low, self.action_high)

    def update(self) -> dict:
        states, actions, rewards, next_states, dones = self.replay.sample(self.cfg.batch_size)
        batch = states.shape[0]

        # 1) Soft Bellman target: r + gamma (1-d) [min(Q'_1,Q'_2) - alpha log pi].
        with np.errstate(over="ignore", invalid="ignore"):
            next_sample = self._actor_sample(next_states)
            next_input = np.concatenate([next_states, next_sample["actions"]], axis=1)
            next_q1, _ = self.q1_target.forward(next_input)
            next_q2, _ = self.q2_target.forward(next_input)
            soft_next_q = np.minimum(next_q1, next_q2) - self.alpha * next_sample["logp"]
            target_q = rewards + self.cfg.gamma * (1.0 - dones) * soft_next_q

        # 2) Fit both critics to the same detached target.
        critic_input = np.concatenate([states, actions], axis=1)
        q1_value, q1_cache = self.q1.forward(critic_input)
        q2_value, q2_cache = self.q2.forward(critic_input)
        q1_loss = 0.5 * float(np.mean((q1_value - target_q) ** 2))
        q2_loss = 0.5 * float(np.mean((q2_value - target_q) ** 2))
        self.q1.backward(q1_cache, (q1_value - target_q) / batch, update=True)
        self.q2.backward(q2_cache, (q2_value - target_q) / batch, update=True)

        # 3) Reparameterized actor objective: E[alpha log pi(a|s) - min_i Q_i(s,a)].
        sample = self._actor_sample(states)
        actor_input = np.concatenate([states, sample["actions"]], axis=1)
        av1, ac1 = self.q1.forward(actor_input)
        av2, ac2 = self.q2.forward(actor_input)
        choose_q1 = (av1 <= av2).astype(np.float64)
        choose_q2 = 1.0 - choose_q1
        grad_in1 = self.q1.backward(ac1, choose_q1 / batch, update=False)
        grad_in2 = self.q2.backward(ac2, choose_q2 / batch, update=False)
        d_q_d_action = grad_in1[:, self.state_dim:] + grad_in2[:, self.state_dim:]

        tanh_jacobian = self.action_scale * (1.0 - sample["squashed"] ** 2)
        alpha = self.alpha
        dmu = (alpha * sample["dlogp_dmu"] / batch
               - d_q_d_action * tanh_jacobian)
        dlogstd = (alpha * sample["dlogp_dlogstd"] / batch
                   - d_q_d_action * tanh_jacobian * sample["std"] * sample["eps"])
        grad_actor_output = np.concatenate([dmu, dlogstd * sample["clip_mask"]], axis=1)
        self.actor.backward(sample["cache"], grad_actor_output, update=True)
        actor_loss = float(np.mean(alpha * sample["logp"] - np.minimum(av1, av2)))

        # 4) Temperature dual update. H_target is positive here by convention:
        # d L / d log(alpha) = -(E[log pi] + H_target) = H(pi) - H_target.
        alpha_grad = -float(np.mean(sample["logp"] + self.target_entropy))
        self.alpha_t += 1
        b1, b2 = 0.9, 0.999
        self.alpha_m = b1 * self.alpha_m + (1.0 - b1) * alpha_grad
        self.alpha_v = b2 * self.alpha_v + (1.0 - b2) * alpha_grad * alpha_grad
        mhat = self.alpha_m / (1.0 - b1 ** self.alpha_t)
        vhat = self.alpha_v / (1.0 - b2 ** self.alpha_t)
        self.log_alpha -= self.cfg.lr * mhat / (math.sqrt(vhat) + 1e-8)
        self.log_alpha = float(np.clip(self.log_alpha, -10.0, 2.0))

        # 5) Polyak averaging of target critics.
        for source, target in ((self.q1, self.q1_target), (self.q2, self.q2_target)):
            for key in source.params:
                target.params[key] *= (1.0 - self.cfg.tau)
                target.params[key] += self.cfg.tau * source.params[key]

        return {"q1_loss": q1_loss, "q2_loss": q2_loss, "actor_loss": actor_loss,
                "alpha": self.alpha, "sample_entropy": -float(np.mean(sample["logp"]))}


## 6. Developing Scientific Controls: Evaluation Metrics

To measure if our agent is actually learning, we must compare it against standard controls. We establish two test scenarios:
1. A completely random controller that acts without thinking, providing our baseline performance.
2. An evaluation run where the agent acts deterministically, using the mean of its distribution ($a = \tanh(\mu)$) rather than sampling stochastically. This isolates what the agent has actually learned from its explorative noise.

In [5]:
def evaluate(agent: SAC, episodes: int = 20, seed: int = 10_000) -> dict:
    env = PointMassEnv(seed=seed)
    returns, lengths, final_abs_x = [], [], []
    for _ in range(episodes):
        state = env.reset()
        total = 0.0
        length = 0
        while True:
            action = agent.act(state, deterministic=True)
            state, reward, terminated, truncated = env.step(action)
            total += reward
            length += 1
            if terminated or truncated:
                break
        returns.append(total)
        lengths.append(length)
        final_abs_x.append(abs(env.x))
    return {"return_mean": float(np.mean(returns)),
            "return_std": float(np.std(returns)),
            "length_mean": float(np.mean(lengths)),
            "final_abs_x_mean": float(np.mean(final_abs_x))}


def evaluate_random(episodes: int = 20, seed: int = 20_000) -> dict:
    env = PointMassEnv(seed=seed)
    rng = np.random.default_rng(seed + 1)
    returns = []
    for _ in range(episodes):
        state = env.reset()
        total = 0.0
        while True:
            state, reward, terminated, truncated = env.step(rng.uniform(-1.0, 1.0, size=1))
            total += reward
            if terminated or truncated:
                break
        returns.append(total)
    return {"return_mean": float(np.mean(returns)), "return_std": float(np.std(returns))}


## 7. The Experimental Training Loop

Now, we orchestrate the training process. The system begins by executing random actions to populate the replay buffer with initial physical dynamics. Once the buffer has enough data, we alternate between stepping through the environment and updating our neural weights.

Every few thousand steps, we pause to evaluate our deterministic policy, saving our metrics to disk to observe the emergent behavior.

In [6]:
def train(steps: int = 15_000, seed: int = 7, output_dir: str = "/mnt/data") -> dict:
    rng = np.random.default_rng(seed)
    env = PointMassEnv(seed=seed)
    config = SACConfig()
    agent = SAC(state_dim=2, action_low=env.action_low, action_high=env.action_high,
                seed=seed + 1, config=config)
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    log_path = output_path / "sac_training_log.csv"
    baseline = evaluate_random(episodes=100, seed=40_000)
    state = env.reset()
    records = []
    last_metrics = {}
    eval_interval = max(1_000, steps // 6)

    with log_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["step", "eval_return_mean", "eval_return_std",
                                               "eval_length_mean", "final_abs_x_mean", "alpha",
                                               "sample_entropy", "q1_loss", "q2_loss", "actor_loss"])
        writer.writeheader()
        for step in range(1, steps + 1):
            if step <= config.learning_starts:
                action = rng.uniform(env.action_low, env.action_high)
            else:
                action = agent.act(state, deterministic=False)
            next_state, reward, terminated, truncated = env.step(action)
            agent.replay.add(state, action, reward, next_state, terminated)
            state = next_state
            if terminated or truncated:
                state = env.reset()

            if step >= config.learning_starts and agent.replay.size >= config.batch_size:
                last_metrics = agent.update()

            if step % eval_interval == 0 or step == steps:
                ev = evaluate(agent, episodes=30, seed=30_000)
                row = {"step": step, "eval_return_mean": ev["return_mean"],
                       "eval_return_std": ev["return_std"], "eval_length_mean": ev["length_mean"],
                       "final_abs_x_mean": ev["final_abs_x_mean"], "alpha": agent.alpha,
                       "sample_entropy": last_metrics.get("sample_entropy", float("nan")),
                       "q1_loss": last_metrics.get("q1_loss", float("nan")),
                       "q2_loss": last_metrics.get("q2_loss", float("nan")),
                       "actor_loss": last_metrics.get("actor_loss", float("nan"))}
                records.append(row)
                writer.writerow(row)
                f.flush()
                print(f"step={step:6d} eval_return={ev['return_mean']:9.3f} +/- {ev['return_std']:.3f} "
                      f"| final_abs_x={ev['final_abs_x_mean']:.3f} | alpha={agent.alpha:.3f}")

    final_eval = evaluate(agent, episodes=100, seed=40_000)
    checkpoint_path = output_path / "sac_numpy_checkpoint.npz"
    checkpoint_arrays = {"log_alpha": np.asarray(agent.log_alpha)}
    for prefix, network in (("actor", agent.actor), ("q1", agent.q1), ("q2", agent.q2),
                            ("q1_target", agent.q1_target), ("q2_target", agent.q2_target)):
        for key, value in network.params.items():
            checkpoint_arrays[f"{prefix}_{key}"] = value
    np.savez_compressed(checkpoint_path, **checkpoint_arrays)
    result = {
        "seed": seed,
        "steps": steps,
        "baseline_random": baseline,
        "final_deterministic_eval_100_episodes": final_eval,
        "training_log": str(log_path),
        "checkpoint": str(checkpoint_path),
        "config": vars(config),
        "last_update_metrics": last_metrics,
        "checkpoints": records,
    }
    import json
    result_path = output_path / "sac_training_results.json"
    result_path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


## 8. Executing the Experiment

We are ready to run the training process. We will train our agent for 6,000 steps using seed 7. Because we are running pure NumPy operations on the CPU, we can watch how efficiently our manual backpropagation updates progress relative to the random baseline.

In [7]:
TRAINING_STEPS = 6_000
SEED = 7
OUTPUT_DIR = Path.cwd() / "sac_outputs"

results = train(steps=TRAINING_STEPS, seed=SEED, output_dir=str(OUTPUT_DIR))

baseline = results["baseline_random"]
trained = results["final_deterministic_eval_100_episodes"]
print("\nSummary")
print(f"Random baseline return: {baseline['return_mean']:.3f} +/- {baseline['return_std']:.3f}")
print(f"Trained deterministic return: {trained['return_mean']:.3f} +/- {trained['return_std']:.3f}")
print(f"Mean final |position|: {trained['final_abs_x_mean']:.4f}")
print(f"Artifacts directory: {OUTPUT_DIR.resolve()}")


step=  1000 eval_return=  -92.040 +/- 94.580 | final_abs_x=0.905 | alpha=0.853
step=  2000 eval_return=  -10.446 +/- 10.745 | final_abs_x=0.076 | alpha=0.653
step=  3000 eval_return=   -7.323 +/- 7.995 | final_abs_x=0.018 | alpha=0.702
step=  4000 eval_return=   -7.265 +/- 7.985 | final_abs_x=0.030 | alpha=0.886
step=  5000 eval_return=   -6.750 +/- 7.359 | final_abs_x=0.034 | alpha=1.049
step=  6000 eval_return=   -6.716 +/- 7.278 | final_abs_x=0.031 | alpha=1.163

Summary
Random baseline return: -98.383 +/- 90.230
Trained deterministic return: -6.996 +/- 6.559
Mean final |position|: 0.0311
Artifacts directory: /content/sac_outputs


## 9. Analyzing the Learning Trajectory

Let's print the training checkpoints. As learning progresses, the average return should trend upward (becoming less negative) while the final position error decreases, indicating that our agent is successfully learning to stabilize the point mass.

In [8]:
print(f"{'step':>8} {'mean return':>14} {'return std':>12} {'final |x|':>12} {'alpha':>10}")
for row in results["checkpoints"]:
    print(f"{int(row['step']):8d} {row['eval_return_mean']:14.3f} "
          f"{row['eval_return_std']:12.3f} {row['final_abs_x_mean']:12.4f} "
          f"{row['alpha']:10.4f}")


    step    mean return   return std    final |x|      alpha
    1000        -92.040       94.580       0.9052     0.8531
    2000        -10.446       10.745       0.0757     0.6530
    3000         -7.323        7.995       0.0181     0.7018
    4000         -7.265        7.985       0.0303     0.8864
    5000         -6.750        7.359       0.0342     1.0491
    6000         -6.716        7.278       0.0313     1.1634


## 10. Verification: Reloading and Testing the Saved Parameters

To ensure our model's parameters were serialized correctly, we will instantiate a completely fresh SAC agent, load our saved parameters from disk, and evaluate its performance. This confirms that our agent can operate purely on the saved weights without any residual state in memory.

In [9]:
env = PointMassEnv(seed=SEED)
loaded_agent = SAC(state_dim=2, action_low=env.action_low, action_high=env.action_high,
                   seed=SEED + 1, config=SACConfig())
loaded_agent.load_checkpoint(str(OUTPUT_DIR / "sac_numpy_checkpoint.npz"))
checkpoint_eval = evaluate(loaded_agent, episodes=100, seed=40_000)
print(f"Reloaded checkpoint return: {checkpoint_eval['return_mean']:.3f} +/- "
      f"{checkpoint_eval['return_std']:.3f}")
print(f"Reloaded checkpoint mean final |position|: {checkpoint_eval['final_abs_x_mean']:.4f}")


Reloaded checkpoint return: -6.996 +/- 6.559
Reloaded checkpoint mean final |position|: 0.0311


## 11. Philosophical Inquiry: Entropy, Choice, and Determinism

Implementing SAC raises fascinating questions about the nature of choice and action. By explicitly rewarding our policy for maintaining entropy, we encourage it to keep options open when several paths have equal value. This offers a compelling analogy to human decision-making: we often value freedom of choice for its own sake, preserving optionality when we are uncertain.

However, we must avoid anthropomorphizing: **stochasticity is not agency**. Sampling from a probability distribution does not equate to self-determination, consciousness, or metaphysical free will. A random agent is not a free agent; indeed, a purely random controller lacks any meaningful control over its destiny.

In our agent, entropy is simply a practical exploration mechanism and a mathematical tool for stability—not a soul. The deeper philosophical questions lie in how intent is formed, how systems respond to environmental feedback, and how complex, adaptive behaviors can emerge from simple, deterministic mathematical updates.

## 12. Future Directions for Investigation

Now that you have built this agent from scratch, here are some experiments to deepen your understanding:
- **Extend the Training**: Increase `TRAINING_STEPS` to 15,000 or 20,000 and observe if the agent's performance reaches a stable plateau.
- **Vary the Seeds**: Run the experiment with different seeds to examine how initial network weights affect stability and convergence.
- **Adjust Exploration**: Change `target_entropy_per_dim` in `SACConfig` to see how forcing higher or lower entropy targets changes the rate of learning.
- **Inspect the Math**: Since we wrote these gradients by hand, any structural change to the networks requires reworking the calculus. Try adding an extra layer and updating the backpropagation equations accordingly!